# Supplier Performance Analysis


## Project Overview

This notebook analyses purchase order data across 10 suppliers to evaluate on-time delivery, defect rates, rejection rates, lead times, and cost variance. A composite supplier tier ranking is derived at the end.


## Learning Objectives

- Compute on-time delivery rates and delay distributions per supplier
- Compare defect and rejection rates across categories
- Rank suppliers using a weighted composite score
- Apply ANOVA to test whether delivery delay differs significantly by supplier


## Business or Research Problem

Procurement teams need an objective, data-driven view of supplier reliability. Without it, poor performers go undetected, costs rise, and production schedules slip.


## Why This Analysis Matters

Supplier performance directly affects inventory availability, product quality, and total landed cost. Identifying weak links enables targeted improvement plans or vendor consolidation.


## Dataset Overview

- **Rows:** 1 500 purchase orders
- **Suppliers:** S001–S010
- **Key columns:** po_id, supplier_id, category, ordered_date, promised_date, actual_date, quantity, unit_cost, defect_rate, rejection_rate
- **Derived:** on_time (bool), delay_days (int)


## Dataset Source and License Notes

This dataset is fully synthetic, generated with NumPy (seed=42). It is provided for educational use only and carries no real-world licensing restrictions.


## Environment Setup

All required libraries are part of the standard data-science stack. Run the cell below to confirm versions.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')


## Configuration / Constants


In [ ]:
SEED = 42
N_ORDERS = 1500
SUPPLIERS = [f'S{str(i).zfill(3)}' for i in range(1, 11)]
CATEGORIES = ['Electronics', 'Packaging', 'Raw Materials', 'Consumables', 'Spare Parts']
FIG_SIZE = (10, 5)


## Dataset Download or Loading

We generate 1 500 synthetic purchase orders with realistic distributions.


In [ ]:
rng = np.random.default_rng(SEED)

po_ids = [f'PO{str(i).zfill(5)}' for i in range(1, N_ORDERS + 1)]
supplier_ids = rng.choice(SUPPLIERS, size=N_ORDERS)
categories = rng.choice(CATEGORIES, size=N_ORDERS)

ordered_dates = pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 365, N_ORDERS), unit='D')
lead_days = rng.integers(5, 30, N_ORDERS)
promised_dates = ordered_dates + pd.to_timedelta(lead_days, unit='D')

# Supplier-specific delay bias
supplier_delay = {'S001': -1, 'S002': 2, 'S003': 5, 'S004': -2, 'S005': 0,
                  'S006': 3, 'S007': 1, 'S008': -1, 'S009': 4, 'S010': 2}
delay_arr = np.array([supplier_delay[s] + rng.integers(-3, 6) for s in supplier_ids])
actual_dates = promised_dates + pd.to_timedelta(delay_arr, unit='D')

quantity = rng.integers(50, 500, N_ORDERS)
unit_cost = rng.uniform(5, 200, N_ORDERS).round(2)

supplier_defect = {'S001': 0.01, 'S002': 0.03, 'S003': 0.06, 'S004': 0.01, 'S005': 0.02,
                   'S006': 0.04, 'S007': 0.02, 'S008': 0.01, 'S009': 0.05, 'S010': 0.03}
defect_rate = np.array([max(0, rng.normal(supplier_defect[s], 0.01)) for s in supplier_ids]).round(4)
rejection_rate = (defect_rate * rng.uniform(0.5, 0.9, N_ORDERS)).round(4)

df = pd.DataFrame({
    'po_id': po_ids, 'supplier_id': supplier_ids, 'category': categories,
    'ordered_date': ordered_dates, 'promised_date': promised_dates,
    'actual_date': actual_dates, 'quantity': quantity, 'unit_cost': unit_cost,
    'defect_rate': defect_rate, 'rejection_rate': rejection_rate
})

df['delay_days'] = (df['actual_date'] - df['promised_date']).dt.days
df['on_time'] = df['delay_days'] <= 0
df['total_cost'] = df['quantity'] * df['unit_cost']
print(df.shape)
df.head(3)


## Data Validation Checks


In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDelay days range:', df['delay_days'].min(), 'to', df['delay_days'].max())
print('Defect rate range:', df['defect_rate'].min(), 'to', df['defect_rate'].max())
assert df['rejection_rate'].max() <= 1.0, 'Rejection rate out of bounds'
print('All validation checks passed.')


## Data Cleaning


In [ ]:
# Clip any negative defect/rejection rates to 0
df['defect_rate'] = df['defect_rate'].clip(lower=0)
df['rejection_rate'] = df['rejection_rate'].clip(lower=0)
print('Data cleaning complete. Shape:', df.shape)


## Exploratory Data Analysis

We begin with overall summary statistics.


In [ ]:
print(df[['delay_days','quantity','unit_cost','defect_rate','rejection_rate']].describe().round(3))
overall_on_time = df['on_time'].mean()
print(f'\nOverall on-time delivery rate: {overall_on_time:.1%}')


## Univariate Analysis

Distribution of delay days and defect rates.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['delay_days'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Delay Days')
axes[0].set_xlabel('Delay Days')
axes[1].hist(df['defect_rate'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Distribution of Defect Rate')
axes[1].set_xlabel('Defect Rate')
plt.tight_layout()
plt.show()


The delay distribution is slightly right-skewed, indicating a minority of severely late orders. Defect rates cluster near zero but some suppliers pull the tail rightward.


## Bivariate / Multivariate Analysis

On-time delivery rate and mean delay days by supplier.


In [ ]:
supplier_summary = df.groupby('supplier_id').agg(
    orders=('po_id', 'count'),
    on_time_rate=('on_time', 'mean'),
    mean_delay=('delay_days', 'mean'),
    mean_defect=('defect_rate', 'mean'),
    mean_rejection=('rejection_rate', 'mean'),
    mean_cost=('unit_cost', 'mean')
).round(4).reset_index()
print(supplier_summary.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)
bars = ax.bar(supplier_summary['supplier_id'], supplier_summary['on_time_rate'] * 100, color='teal')
ax.axhline(overall_on_time * 100, color='red', linestyle='--', label=f'Overall avg {overall_on_time:.1%}')
ax.set_title('On-Time Delivery Rate by Supplier')
ax.set_ylabel('On-Time Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()


Suppliers S004 and S001 are top performers; S003 and S009 lag significantly.


## Feature-Specific Insights

Defect and rejection rate comparison by supplier.


In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)
x = np.arange(len(supplier_summary))
width = 0.4
ax.bar(x - width/2, supplier_summary['mean_defect'] * 100, width, label='Defect Rate %', color='coral')
ax.bar(x + width/2, supplier_summary['mean_rejection'] * 100, width, label='Rejection Rate %', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(supplier_summary['supplier_id'])
ax.set_title('Defect vs Rejection Rate by Supplier')
ax.set_ylabel('Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()


## Statistical Checks

One-way ANOVA tests whether mean delivery delay differs significantly across suppliers.


In [ ]:
groups = [group['delay_days'].values for _, group in df.groupby('supplier_id')]
f_stat, p_value = stats.f_oneway(*groups)
print(f'ANOVA F-statistic: {f_stat:.2f}, p-value: {p_value:.4f}')
if p_value < 0.05:
    print('Result: Delivery delay differs significantly across suppliers (p < 0.05).')
else:
    print('Result: No significant difference in delivery delay across suppliers.')


## Visual Analysis

Heatmap of mean delay days by supplier and category.


In [ ]:
pivot = df.pivot_table(values='delay_days', index='supplier_id', columns='category', aggfunc='mean').round(1)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax)
ax.set_title('Mean Delay Days by Supplier and Category')
plt.tight_layout()
plt.show()


Darker cells highlight problematic supplier-category combinations that warrant targeted intervention.

## Supplier Tier Ranking

We compute a composite score using on-time rate, defect rate, and rejection rate.


In [ ]:
# Normalize metrics (0-1, higher is better)
s = supplier_summary.copy()
s['norm_ontime'] = (s['on_time_rate'] - s['on_time_rate'].min()) / (s['on_time_rate'].max() - s['on_time_rate'].min())
s['norm_defect'] = 1 - (s['mean_defect'] - s['mean_defect'].min()) / (s['mean_defect'].max() - s['mean_defect'].min())
s['norm_rejection'] = 1 - (s['mean_rejection'] - s['mean_rejection'].min()) / (s['mean_rejection'].max() - s['mean_rejection'].min())
s['composite_score'] = (0.5 * s['norm_ontime'] + 0.3 * s['norm_defect'] + 0.2 * s['norm_rejection']).round(4)
s = s.sort_values('composite_score', ascending=False).reset_index(drop=True)
s['tier'] = pd.cut(s['composite_score'], bins=[0, 0.4, 0.7, 1.01], labels=['C', 'B', 'A'])
print(s[['supplier_id', 'on_time_rate', 'mean_defect', 'composite_score', 'tier']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = {'A': 'green', 'B': 'orange', 'C': 'red'}
bar_colors = [colors[t] for t in s['tier'].astype(str)]
ax.barh(s['supplier_id'], s['composite_score'], color=bar_colors)
ax.set_title('Composite Supplier Score (A=green, B=orange, C=red)')
ax.set_xlabel('Composite Score')
plt.tight_layout()
plt.show()
best = s.iloc[0]
worst = s.iloc[-1]
print(f'Top supplier: {best["supplier_id"]} (score {best["composite_score"]:.3f}, Tier {best["tier"]})')
print(f'Worst supplier: {worst["supplier_id"]} (score {worst["composite_score"]:.3f}, Tier {worst["tier"]})')


## Key Findings


In [ ]:
print(f'1. Overall on-time delivery rate: {overall_on_time:.1%}')
print(f'2. ANOVA confirms delivery delay differs by supplier (F={f_stat:.2f}, p={p_value:.4f})')
print(f'3. Best composite supplier: {best["supplier_id"]} | Worst: {worst["supplier_id"]}')
tier_a = (s['tier'] == 'A').sum()
tier_c = (s['tier'] == 'C').sum()
print(f'4. Tier A (preferred) suppliers: {tier_a} | Tier C (at-risk) suppliers: {tier_c}')


## Limitations

- Synthetic data may not capture seasonal or contract-specific delays
- Cost variance analysis is simplified; actual landed cost includes freight and duties
- ANOVA assumes normally distributed residuals; real data may require Kruskal-Wallis


## Recommendations / Next Steps

1. Place Tier-C suppliers on a performance improvement plan with quarterly reviews
2. Diversify sourcing away from high-defect suppliers for critical categories
3. Negotiate SLA penalties for repeat late deliveries
4. Track composite scores monthly for trend monitoring


## Common Mistakes

- Ranking suppliers by a single metric (e.g., price alone) ignores quality and reliability
- Ignoring volume weighting when comparing defect rates can mislead analysis
- Treating on-time as binary without capturing severity of lateness loses information


## Mini Challenge / Exercises

1. Add a `cost_variance` column (actual vs budgeted cost) and include it in the composite score
2. Segment analysis by `category` — does S003 perform poorly across all categories or only some?
3. Apply Tukey's HSD post-hoc test to identify which supplier pairs differ significantly in delay


## Final Summary / Key Takeaways

- Delivery performance varies significantly across suppliers, confirmed by ANOVA
- Defect and rejection rates are correlated but not identical — rejection may be caught inspection-side
- Composite scoring provides a balanced view; procurement teams should use it alongside qualitative vendor reviews
- Tier-A suppliers should receive preferred-vendor status and larger order volumes
